In [1]:
import os
import sys

# 현재 작업 디렉토리 기준으로 상위 1단계 폴더를 루트로 설정
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# print("프로젝트 루트로 설정된 경로:", project_root)

In [2]:
# 데이터 확인하기 2025.11.21
# 이상인 컬럼 제거 후 RandomForest 기본 모델 돌리기
import pandas as pd
import numpy  as np
from scipy.special import logit
import time
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler # 데이터 전처리용
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix, recall_score, precision_score
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

import matplotlib.pyplot as plt
import seaborn as sns

import importlib
from utils import preprocessing

# 모듈 reload
importlib.reload(preprocessing)
# importlib.reload(user_utils)

from utils.preprocessing import load_data, split_features_target, scale_data, data_split, remove_zero_columns
from utils.user_utils    import get_model_train_eval
from utils.model_utils   import save_model, load_model
from utils.evaluation import evaluate_model_cv

# train = pd.read_csv("../data/train.csv")
# test  = pd.read_csv("../data/test.csv")

In [3]:
# ============================================================
# 1. 데이터 로드 + ID / TARGET 제거
# ============================================================
print("="*100)
print("Step1. 데이터 로드 + ID/TARGET 제거")
print("="*100)

DATA_DIR = os.path.join(project_root, "data")
DOC_DIR  = os.path.join(project_root, "doc")

train_path = os.path.join(DATA_DIR, "train.csv")
test_path  = os.path.join(DATA_DIR, "test.csv")

train_raw = pd.read_csv(train_path)
test_raw  = pd.read_csv(test_path)

print(f"train_raw shape: {train_raw.shape}")
print(f"test_raw  shape: {test_raw.shape}")
print(train_raw.head(3))

ID_COL = "ID"
TARGET_COL = "TARGET"

# y, X, X_test 분리
y = train_raw[TARGET_COL].copy()
X = train_raw.drop(columns=[ID_COL, TARGET_COL]).copy()
X_test = test_raw.drop(columns=[ID_COL]).copy()

print("\nID / TARGET 제거 후")
print(f"X shape      : {X.shape}")
print(f"X_test shape : {X_test.shape}")
print(f"y shape      : {y.shape}")
print(f"양성 비율(TARGET=1 비율): {y.mean():.4f}")


Step1. 데이터 로드 + ID/TARGET 제거
train_raw shape: (76020, 371)
test_raw  shape: (75818, 370)
   ID  var3  var15  imp_ent_var16_ult1  imp_op_var39_comer_ult1  \
0   1     2     23                 0.0                      0.0   
1   3     2     34                 0.0                      0.0   
2   4     2     23                 0.0                      0.0   

   imp_op_var39_comer_ult3  imp_op_var40_comer_ult1  imp_op_var40_comer_ult3  \
0                      0.0                      0.0                      0.0   
1                      0.0                      0.0                      0.0   
2                      0.0                      0.0                      0.0   

   imp_op_var40_efect_ult1  imp_op_var40_efect_ult3  ...  \
0                      0.0                      0.0  ...   
1                      0.0                      0.0  ...   
2                      0.0                      0.0  ...   

   saldo_medio_var33_hace2  saldo_medio_var33_hace3  saldo_medio_var33_ult1  \
0

In [4]:
# ============================================================
# 2. Zero Count > 99% 컬럼 제거 (remove_cols_0.99.txt)
# ============================================================
print("\n" + "="*100)
print("Step2. Zero Count 기반 컬럼 제거 (remove_cols_0.99.txt)")
print("="*100)

zero_file = os.path.join(DOC_DIR, "remove_cols_0.99.txt")
zero_cols_raw = []

with open(zero_file, "r", encoding="utf-8") as f:
    for line in f:
        col = line.strip()
        if col:
            zero_cols_raw.append(col)

print(f"파일에서 읽은 제거 대상 컬럼 수: {len(zero_cols_raw)}")

zero_cols_in_X     = [c for c in zero_cols_raw if c in X.columns]
zero_cols_not_in_X = [c for c in zero_cols_raw if c not in X.columns]

print(f"Train에 실제 존재하는 제거 컬럼 수: {len(zero_cols_in_X)}")
if zero_cols_not_in_X:
    print("현재 데이터에 없는 컬럼 예시:", zero_cols_not_in_X[:10])

X1      = X.drop(columns=zero_cols_in_X)
X_test1 = X_test.drop(columns=zero_cols_in_X, errors="ignore")

print("\nZero Count 제거 후")
print(f"Train: {X.shape} → {X1.shape}")
print(f"Test : {X_test.shape} → {X_test1.shape}")

X_current      = X1.copy()
X_test_current = X_test1.copy()



Step2. Zero Count 기반 컬럼 제거 (remove_cols_0.99.txt)
파일에서 읽은 제거 대상 컬럼 수: 217
Train에 실제 존재하는 제거 컬럼 수: 217

Zero Count 제거 후
Train: (76020, 369) → (76020, 152)
Test : (75818, 369) → (75818, 152)


In [5]:
# ============================================================
# 3. var3 결측치(-999999) → 최빈값 2로 치환
# ============================================================
print("\n" + "="*100)
print("Step3. var3 결측치(-999999) 치환")
print("="*100)

for df, name in [(X_current, "X_current"), (X_test_current, "X_test_current")]:
    if "var3" in df.columns:
        before_cnt = (df["var3"] == -999999).sum()
        df["var3"] = df["var3"].replace(-999999, 2)
        after_cnt = (df["var3"] == -999999).sum()
        print(f"{name} - var3 -999999 개수: {before_cnt} → {after_cnt}")
    else:
        print(f"{name} 에 var3 컬럼이 없습니다.")

print("var3 처리 후 shape:")
print("X_current     :", X_current.shape)
print("X_test_current:", X_test_current.shape)



Step3. var3 결측치(-999999) 치환
X_current - var3 -999999 개수: 116 → 0
X_test_current - var3 -999999 개수: 120 → 0
var3 처리 후 shape:
X_current     : (76020, 152)
X_test_current: (75818, 152)


In [6]:
# ============================================================
# 4. 상관계수 > 0.95 컬럼 제거 (remove_cols_train_0.95.txt)
# ============================================================
print("\n" + "="*100)
print("Step4. 상관계수 0.95 이상 컬럼 제거 (remove_cols_train_0.95.txt)")
print("="*100)

corr_file = os.path.join(DOC_DIR, "remove_cols_train_0.95.txt")
corr_cols_raw = []

with open(corr_file, "r", encoding="utf-8") as f:
    for line in f:
        col = line.strip()
        if col:
            corr_cols_raw.append(col)

print(f"파일에서 읽은 상관계수 제거 대상 컬럼 수: {len(corr_cols_raw)}")

corr_cols_in_X     = [c for c in corr_cols_raw if c in X_current.columns]
corr_cols_not_in_X = [c for c in corr_cols_raw if c not in X_current.columns]

print(f"Train에 실제 존재하는 제거 컬럼 수: {len(corr_cols_in_X)}")
if corr_cols_not_in_X:
    print("현재 데이터에 없는 컬럼 예시:", corr_cols_not_in_X[:10])

X2      = X_current.drop(columns=corr_cols_in_X)
X_test2 = X_test_current.drop(columns=corr_cols_in_X, errors="ignore")

print("\n상관계수 제거 후")
print(f"Train: {X_current.shape} → {X2.shape}")
print(f"Test : {X_test_current.shape} → {X_test2.shape}")
print("실제로 삭제된 컬럼 개수:", len(corr_cols_in_X))

X_current      = X2.copy()
X_test_current = X_test2.copy()



Step4. 상관계수 0.95 이상 컬럼 제거 (remove_cols_train_0.95.txt)
파일에서 읽은 상관계수 제거 대상 컬럼 수: 283
Train에 실제 존재하는 제거 컬럼 수: 66
현재 데이터에 없는 컬럼 예시: ['imp_op_var40_comer_ult1', 'imp_op_var40_comer_ult3', 'imp_op_var40_efect_ult1', 'imp_op_var40_efect_ult3', 'imp_op_var40_ult1', 'imp_sal_var16_ult1', 'ind_var1', 'ind_var2_0', 'ind_var2', 'ind_var6_0']

상관계수 제거 후
Train: (76020, 152) → (76020, 86)
Test : (75818, 152) → (75818, 86)
실제로 삭제된 컬럼 개수: 66


In [7]:
# ============================================================
# 5. Log1p 변환 (Log1pColumns.txt – 이상치/왜도 큰 컬럼 71개)
# ============================================================
print("\n" + "="*100)
print("Step5. Log1p 변환 (Log1pColumns.txt)")
print("="*100)

log_file = os.path.join(DOC_DIR, "Log1pColumns.txt")
log_cols_raw = []

with open(log_file, "r", encoding="utf-8") as f:
    for line in f:
        col = line.strip()
        if col:
            log_cols_raw.append(col)

print(f"Log1p 파일에서 읽은 전체 대상 컬럼 수: {len(log_cols_raw)}")

log_cols_in_X     = [c for c in log_cols_raw if c in X_current.columns]
log_cols_not_in_X = [c for c in log_cols_raw if c not in X_current.columns]

print(f"실제 Log1p 적용 가능한 컬럼 수: {len(log_cols_in_X)}")
if log_cols_not_in_X:
    print("현재 데이터에 없는 Log1p 컬럼 예시:", log_cols_not_in_X[:10])

for col in log_cols_in_X:
    X_current[col]      = np.log1p(X_current[col])
    X_test_current[col] = np.log1p(X_test_current[col])

print("\nLog1p 변환 적용 완료")
print("X_current shape     :", X_current.shape)
print("X_test_current shape:", X_test_current.shape)



Step5. Log1p 변환 (Log1pColumns.txt)
Log1p 파일에서 읽은 전체 대상 컬럼 수: 72
실제 Log1p 적용 가능한 컬럼 수: 41
현재 데이터에 없는 Log1p 컬럼 예시: ['Log1p_Colums', 'num_var14_0', 'imp_ent_var16_ult1', 'saldo_var26', 'saldo_medio_var12_hace3', 'saldo_medio_var12_hace2', 'num_op_var41_hace3', 'num_trasp_var11_ult1', 'saldo_medio_var13_corto_hace3', 'var21']

Log1p 변환 적용 완료
X_current shape     : (76020, 86)
X_test_current shape: (75818, 86)


In [8]:
# ============================================================
# 6. Train / Valid 분리 + StandardScaler
# ============================================================
print("\n" + "="*100)
print("Step6. Train / Valid 분리 + StandardScaler")
print("="*100)

X_train, X_val, y_train, y_val = train_test_split(
    X_current,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_val   shape: {X_val.shape}")
print(f"y_train mean : {y_train.mean():.4f}")
print(f"y_val   mean : {y_val.mean():.4f}")

scaler = StandardScaler()

X_train_scaled_np = scaler.fit_transform(X_train)
X_val_scaled_np   = scaler.transform(X_val)
X_test_scaled_np  = scaler.transform(X_test_current)

X_train_scaled = pd.DataFrame(X_train_scaled_np, columns=X_train.columns, index=X_train.index)
X_val_scaled   = pd.DataFrame(X_val_scaled_np,   columns=X_val.columns,   index=X_val.index)
X_test_scaled  = pd.DataFrame(X_test_scaled_np,  columns=X_test_current.columns, index=X_test_current.index)

print("\n스케일링 후")
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_val_scaled   shape:", X_val_scaled.shape)
print("X_test_scaled  shape:", X_test_scaled.shape)



Step6. Train / Valid 분리 + StandardScaler
X_train shape: (60816, 86)
X_val   shape: (15204, 86)
y_train mean : 0.0396
y_val   mean : 0.0396

스케일링 후
X_train_scaled shape: (60816, 86)
X_val_scaled   shape: (15204, 86)
X_test_scaled  shape: (75818, 86)


In [ ]:
# ============================================================
# 7. 평가 함수 정의 (공통 사용)
# ============================================================
def print_full_metrics(model_name, y_true, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)
    pre = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_proba)
    cm  = confusion_matrix(y_true, y_pred)
    cr  = classification_report(y_true, y_pred, digits=4)

    print(f"===== {model_name} 성능 (Threshold: {threshold:.2f}) =====")
    print(f"AUC       : {auc:.4f}")
    print(f"정확도    : {acc:.4f}")
    print(f"정밀도    : {pre:.4f}")
    print(f"재현율    : {rec:.4f}")
    print(f"F1-score  : {f1:.4f}")
    print("\nConfusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(cr)
    print()

def find_best_threshold(y_true, y_proba, thresholds=np.arange(0.01, 0.80, 0.01)):
    best_thr = 0.5
    best_f1  = -1
    best_rec = 0.0

    for thr in thresholds:
        y_pred = (y_proba >= thr).astype(int)
        f1  = f1_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1  = f1
            best_thr = thr
            best_rec = rec
    return best_thr, best_f1, best_rec


In [12]:
rng = np.random.default_rng(42)

In [ ]:
# ============================================================
# 8. HyperOpt – RandomForest 튜닝
# ============================================================
print("\n" + "="*100)
print("Step8. HyperOpt – RandomForest")
print("="*100)

# 클래스 불균형 정도 참고
pos = y_train.sum()
neg = len(y_train) - pos
print(f"Train 양성:{pos}, 음성:{neg}, 양성비율:{pos/len(y_train):.4f}")

space_rf = {
    'n_estimators':     hp.quniform('rf_n_estimators', 200, 800, 50),
    'max_depth':        hp.quniform('rf_max_depth', 3, 15, 1),
    'min_samples_split':hp.quniform('rf_min_samples_split', 2, 10, 1),
    'min_samples_leaf': hp.quniform('rf_min_samples_leaf', 1, 10, 1),
    'max_features':     hp.choice('rf_max_features', ['sqrt', 'log2', 0.5, 0.7, 1.0]),
}

def objective_rf(params):
    params = params.copy()
    params['n_estimators']      = int(params['n_estimators'])
    params['max_depth']         = int(params['max_depth'])
    params['min_samples_split'] = int(params['min_samples_split'])
    params['min_samples_leaf']  = int(params['min_samples_leaf'])

    model = RandomForestClassifier(
        n_jobs=-1,
        random_state=42,
        class_weight='balanced',
        **params
    )
    model.fit(X_train_scaled, y_train)
    proba = model.predict_proba(X_val_scaled)[:, 1]
    pred  = (proba >= 0.5).astype(int)
    f1    = f1_score(y_val, pred, zero_division=0)
    return {'loss': -f1, 'status': STATUS_OK}

trials_rf = Trials()
best_rf = fmin(
    fn=objective_rf,
    space=space_rf,
    algo=tpe.suggest,
    max_evals=50,
    trials=trials_rf,
    rstate=rng
)

# 하이퍼파라미터 후처리
best_rf['n_estimators']      = int(best_rf['rf_n_estimators'])
best_rf['max_depth']         = int(best_rf['rf_max_depth'])
best_rf['min_samples_split'] = int(best_rf['rf_min_samples_split'])
best_rf['min_samples_leaf']  = int(best_rf['rf_min_samples_leaf'])
best_rf['max_features']      = ['sqrt', 'log2', 0.5, 0.7, 1.0][best_rf['rf_max_features']]

# rf_ prefix 제거
for k in list(best_rf.keys()):
    if k.startswith('rf_'):
        del best_rf[k]

print("\n[RF] HyperOpt Best Params:")
print(best_rf)

rf_best = RandomForestClassifier(
    n_jobs=-1,
    random_state=42,
    class_weight='balanced',
    **best_rf
)
rf_best.fit(X_train_scaled, y_train)
rf_proba_val = rf_best.predict_proba(X_val_scaled)[:, 1]

print("\n[RF] Validation 성능 (기본 threshold=0.5)")
print_full_metrics("RandomForest (HyperOpt)", y_val, rf_proba_val, threshold=0.5)

rf_best_thr, rf_best_f1, rf_best_rec = find_best_threshold(y_val, rf_proba_val)
print(f"[RF] F1 기준 최적 Threshold: {rf_best_thr:.2f}, F1: {rf_best_f1:.4f}, Recall: {rf_best_rec:.4f}")


Step8. HyperOpt – RandomForest
Train 양성:2406, 음성:58410, 양성비율:0.0396
100%|██████████| 50/50 [17:56<00:00, 21.52s/trial, best loss: -0.26328125]        

[RF] HyperOpt Best Params:
{'n_estimators': 550, 'max_depth': 15, 'min_samples_split': 5, 'min_samples_leaf': 10, 'max_features': 0.7}

[RF] Validation 성능 (기본 threshold=0.5)
===== RandomForest (HyperOpt) 성능 (Threshold: 0.50) =====
AUC       : 0.8384
정확도    : 0.8760
정밀도    : 0.1721
재현율    : 0.5598
F1-score  : 0.2633

Confusion Matrix:
[[12981  1621]
 [  265   337]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9800    0.8890    0.9323     14602
           1     0.1721    0.5598    0.2633       602

    accuracy                         0.8760     15204
   macro avg     0.5761    0.7244    0.5978     15204
weighted avg     0.9480    0.8760    0.9058     15204


[RF] F1 기준 최적 Threshold: 0.67, F1: 0.2760, Recall: 0.4502


In [ ]:
# ============================================================
# 9. HyperOpt – XGBoost 튜닝 (수정 버전)
# ============================================================
print("\n" + "="*100)
print("Step9. HyperOpt – XGBoost")
print("="*100)

# 클래스 불균형 보정용 scale_pos_weight
scale_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
print(f"XGB scale_pos_weight: {scale_pos_weight:.2f}")

# 1) HyperOpt 검색 공간
#    👉 여기서는 'hp.*' 라벨이랑 key 이름을 똑같이 맞춰서 혼동 없게 함
space_xgb = {
    'n_estimators':      hp.quniform('n_estimators',      300, 800, 50),
    'max_depth':         hp.quniform('max_depth',         3,   8,   1),
    'learning_rate':     hp.loguniform('learning_rate',   np.log(0.01), np.log(0.2)),
    'min_child_weight':  hp.quniform('min_child_weight',  1,   10,  1),
    'subsample':         hp.uniform('subsample',          0.6, 1.0),
    'colsample_bytree':  hp.uniform('colsample_bytree',   0.6, 1.0),
    'gamma':             hp.uniform('gamma',              0.0, 5.0),
    'reg_lambda':        hp.loguniform('reg_lambda',      np.log(1e-3), np.log(10)),
}

# 2) objective 함수: F1 최대화 (loss = -F1)
def objective_xgb(params):
    params = params.copy()

    # 정수형으로 써야 하는 애들만 int로 변환
    int_params = ['n_estimators', 'max_depth', 'min_child_weight']
    for p in int_params:
        params[p] = int(params[p])

    model = XGBClassifier(
        objective        = 'binary:logistic',
        eval_metric      = 'logloss',
        tree_method      = 'hist',
        n_jobs           = -1,
        random_state     = 42,
        scale_pos_weight = scale_pos_weight,
        **params
    )

    model.fit(X_train_scaled, y_train)
    proba = model.predict_proba(X_val_scaled)[:, 1]
    pred  = (proba >= 0.5).astype(int)
    f1    = f1_score(y_val, pred, zero_division=0)

    # HyperOpt는 loss를 "최소화"하니까 -F1
    return {'loss': -f1, 'status': STATUS_OK}

# 3) HyperOpt 실행
trials_xgb = Trials()
best_xgb = fmin(
    fn          = objective_xgb,
    space       = space_xgb,
    algo        = tpe.suggest,
    max_evals   = 50,
    trials      = trials_xgb,
    rstate      = rng     # ✅ RandomState 말고 default_rng 사용
)

# 4) fmin 결과 후처리 (정수형 캐스팅 한 번 더 안전하게)
best_xgb['n_estimators']     = int(best_xgb['n_estimators'])
best_xgb['max_depth']        = int(best_xgb['max_depth'])
best_xgb['min_child_weight'] = int(best_xgb['min_child_weight'])

print("\n[XGB] HyperOpt Best Params:")
print(best_xgb)

# 5) 최종 XGB 모델 학습 (train/val 기준)
xgb_best = XGBClassifier(
    objective        = 'binary:logistic',
    eval_metric      = 'logloss',
    tree_method      = 'hist',
    n_jobs           = -1,
    random_state     = 42,
    scale_pos_weight = scale_pos_weight,
    **best_xgb
)

xgb_best.fit(X_train_scaled, y_train)
xgb_proba_val = xgb_best.predict_proba(X_val_scaled)[:, 1]

print("\n[XGB] Validation 성능 (기본 threshold=0.5)")
print_full_metrics("XGBoost (HyperOpt)", y_val, xgb_proba_val, threshold=0.5)

# 6) F1 기준 최적 Threshold 탐색
xgb_best_thr, xgb_best_f1, xgb_best_rec = find_best_threshold(y_val, xgb_proba_val)
print(f"[XGB] F1 기준 최적 Threshold: {xgb_best_thr:.2f}, F1: {xgb_best_f1:.4f}, Recall: {xgb_best_rec:.4f}")



Step9. HyperOpt – XGBoost
XGB scale_pos_weight: 24.28
100%|██████████| 50/50 [02:13<00:00,  2.67s/trial, best loss: -0.2573940847322142]

[XGB] HyperOpt Best Params:
{'colsample_bytree': np.float64(0.9762817499911767), 'gamma': np.float64(1.933229499206568), 'learning_rate': np.float64(0.08492564022420714), 'max_depth': 7, 'min_child_weight': 5, 'n_estimators': 550, 'reg_lambda': np.float64(0.0017352229009395953), 'subsample': np.float64(0.8961982503255849)}

[XGB] Validation 성능 (기본 threshold=0.5)
===== XGBoost (HyperOpt) 성능 (Threshold: 0.50) =====
AUC       : 0.8129
정확도    : 0.8778
정밀도    : 0.1695
재현율    : 0.5349
F1-score  : 0.2574

Confusion Matrix:
[[13024  1578]
 [  280   322]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9790    0.8919    0.9334     14602
           1     0.1695    0.5349    0.2574       602

    accuracy                         0.8778     15204
   macro avg     0.5742    0.7134    0.5954     15204
weighted avg

In [20]:
# ============================================================
# 10. HyperOpt – LightGBM 튜닝 (class_weight까지 최적화)
# ============================================================
print("\n" + "="*100)
print("Step10. HyperOpt – LightGBM (with class_weight search)")
print("="*100)

import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger("lightgbm").setLevel(logging.ERROR)

# 👉 class_weight 후보 (양성 클래스 1의 가중치)
class_weight_candidates = [1.0, 2.0, 3.0, 5.0, 8.0, 10.0, 15.0]

# 1) HyperOpt 검색 공간
space_lgb = {
    'n_estimators':      hp.quniform('n_estimators',      300, 1000, 50),
    'learning_rate':     hp.loguniform('learning_rate',   np.log(0.01), np.log(0.2)),
    'num_leaves':        hp.quniform('num_leaves',        16,  128, 1),
    'min_child_samples': hp.quniform('min_child_samples', 10,  100, 5),
    'subsample':         hp.uniform('subsample',          0.6, 1.0),
    'colsample_bytree':  hp.uniform('colsample_bytree',   0.6, 1.0),
    'reg_lambda':        hp.loguniform('reg_lambda',      np.log(1e-3), np.log(10)),
    # ✅ class_weight의 양성 클래스 가중치를 choice로 탐색
    'class_weight_pos':  hp.choice('class_weight_pos', class_weight_candidates),
}

def objective_lgb(params):
    params = params.copy()

    # 1) 정수형 파라미터 캐스팅
    int_params = ['n_estimators', 'num_leaves', 'min_child_samples']
    for p in int_params:
        params[p] = int(params[p])

    # 2) class_weight 구성
    pos_w = params.pop('class_weight_pos')   # hp.choice → 실제 값이 들어옴 (예: 5.0)
    cw = {0: 1.0, 1: float(pos_w)}           # 음성=1.0 고정, 양성만 튜닝

    model = LGBMClassifier(
        objective    = 'binary',
        n_jobs       = -1,
        random_state = 42,
        verbosity    = -1,
        class_weight = cw,     # ✅ 튜닝된 class_weight 사용
        **params
    )

    model.fit(X_train_scaled, y_train)
    proba = model.predict_proba(X_val_scaled)[:, 1]
    pred  = (proba >= 0.5).astype(int)
    f1    = f1_score(y_val, pred, zero_division=0)

    return {'loss': -f1, 'status': STATUS_OK}

trials_lgb = Trials()
best_lgb = fmin(
    fn        = objective_lgb,
    space     = space_lgb,
    algo      = tpe.suggest,
    max_evals = 50,
    trials    = trials_lgb,
    rstate    = rng   # ← 너가 위에서 쓰고 있는 np.random.default_rng(42)
)

# ------------------------------------------------------------
# fmin 결과 후처리
#   - hp.choice 때문에 best_lgb['class_weight_pos']는 "인덱스"로 들어옴
#   - 우리는 다시 실제 가중치 값으로 치환해줘야 함
# ------------------------------------------------------------
best_lgb['n_estimators']      = int(best_lgb['n_estimators'])
best_lgb['num_leaves']        = int(best_lgb['num_leaves'])
best_lgb['min_child_samples'] = int(best_lgb['min_child_samples'])

# choice 인덱스를 실제 값으로 변환
best_lgb['class_weight_pos'] = class_weight_candidates[best_lgb['class_weight_pos']]

print("\n[LGB] HyperOpt Best Params (with class_weight):")
print(best_lgb)

# 최종 class_weight 구성
final_cw = {0: 1.0, 1: float(best_lgb['class_weight_pos'])}

# **best_lgb에 class_weight_pos는 더 이상 필요 없으니 제거
best_lgb_for_model = best_lgb.copy()
best_lgb_for_model.pop('class_weight_pos')

# 최종 LGBM 모델 학습
lgb_best = LGBMClassifier(
    objective    = 'binary',
    n_jobs       = -1,
    random_state = 42,
    verbosity    = -1,
    class_weight = final_cw,    # ✅ 최종 튜닝된 class_weight 사용
    **best_lgb_for_model
)
lgb_best.fit(X_train_scaled, y_train)
lgb_proba_val = lgb_best.predict_proba(X_val_scaled)[:, 1]

print("\n[LGB] Validation 성능 (기본 threshold=0.5)")
print_full_metrics("LightGBM (HyperOpt + class_weight search)", y_val, lgb_proba_val, threshold=0.5)

lgb_best_thr, lgb_best_f1, lgb_best_rec = find_best_threshold(y_val, lgb_proba_val)
print(f"[LGB] F1 기준 최적 Threshold: {lgb_best_thr:.2f}, F1: {lgb_best_f1:.4f}, Recall: {lgb_best_rec:.4f}")
print(f"[LGB] 최종 class_weight: {final_cw}")



Step10. HyperOpt – LightGBM (with class_weight search)
100%|██████████| 50/50 [03:28<00:00,  4.18s/trial, best loss: -0.29548834903321763]

[LGB] HyperOpt Best Params (with class_weight):
{'class_weight_pos': 8.0, 'colsample_bytree': np.float64(0.92260409157331), 'learning_rate': np.float64(0.023724438768490733), 'min_child_samples': 100, 'n_estimators': 700, 'num_leaves': 75, 'reg_lambda': np.float64(0.7748486630395964), 'subsample': np.float64(0.894326965105685)}

[LGB] Validation 성능 (기본 threshold=0.5)
===== LightGBM (HyperOpt + class_weight search) 성능 (Threshold: 0.50) =====
AUC       : 0.8340
정확도    : 0.9065
정밀도    : 0.2106
재현율    : 0.4950
F1-score  : 0.2955

Confusion Matrix:
[[13485  1117]
 [  304   298]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9780    0.9235    0.9499     14602
           1     0.2106    0.4950    0.2955       602

    accuracy                         0.9065     15204
   macro avg     0.5943    0.7093   

In [ ]:


# 모델 앙상블

# 1. Base models 정의
# 최적 하이퍼파라미터:
rf_best_param = {
    "random_state": 23,
    "n_estimators": 390,
    "max_depth": 25,
    "class_weight": {0: 1, 1: 2},
    "min_samples_leaf": 1,
    "min_samples_split": 7,
    "n_jobs": -1
}
rf_clf = RandomForestClassifier(**rf_best_param)

# 최적 하이퍼파라미터:
xgb_best_params = {
    "random_state": 23,
    "n_estimators": 320,
    "colsample_bytree": 0.88,
    "gamma": 0.058,
    "learning_rate": 0.13,
    "max_depth": 6,
    "scale_pos_weight": 10,
    "min_child_weight": 2,
    "subsample": 0.85,
    "eval_metric": "auc",
    "use_label_encoder": False,
    "n_jobs": -1,
}

xgb_clf = XGBClassifier(**xgb_best_params)
    # random_state      = 23,
    # n_estimators      = 320,
    # colsample_bytree  = 0.88,
    # gamma             = 0.058,
    # learning_rate     = 0.13,
    # max_depth         = 6,
    # scale_pos_weight  = 10,
    # min_child_weight  = 2,
    # subsample         = 0.85,
    # eval_metric       ='auc',
    # use_label_encoder = False,
    # n_jobs            = -1,

# 최적 하이퍼파라미터:
lgbm_best_param = {
    'random_state' : 23,
    'n_estimators' : 400,
    'num_leaves' : 36,
    'learning_rate' : 0.03,
    'subsample' : 0.9,
    'colsample_bytree' : 0.75,
    'reg_alpha' : 0.6,
    'reg_lambda' : 0.2,
    'class_weight' : {0:1, 1:10},
    'n_jobs' : -1

}
# {
#     'random_state' : 23,
#     'n_estimators' : 400,
#     'num_leaves' : 36,
#     'learning_rate' : 0.03,
#     'subsample' : 0.9,
#     'colsample_bytree' : 0.75,
#     'reg_alpha' : 0.6,
#     'reg_lambda' : 0.2,
#     'class_weight' : {0:1, 1:10},
#     'n_jobs' : -1
# }
# lgbm_best_param_org = {
#     random_state     = 23,
#     n_estimators     = 300,
#     colsample_bytree = 0.73,
#     # max_depth      = -1,
#     learning_rate    = 0.03,
#     num_leaves       = 42,
#     reg_alpha        = 0.61,
#     reg_lambda       = 0.13,
#     subsample        = 0.97,
#     class_weight     = {0:1, 1:10},
#     n_jobs           = -1,
# }

lgbm_clf = LGBMClassifier(**lgbm_best_param)


# 2. Meta model 정의 - 윤지훈님 best param.
meta_model = LogisticRegression(
    random_state = 23,
    max_iter     = 1000,
    C            = 0.029,
    penalty      = 'l2',
    solver       = 'lbfgs',
    class_weight ="balanced",
    n_jobs=-1
)
# 3. StackingClassifier 구성
stacking_model = StackingClassifier(
    estimators      = [('rf', rf_clf), ('xgb', xgb_clf), ('lgbm', lgbm_clf)],
    final_estimator = meta_model,
    cv              = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs          = -1
)

# 4. 학습 (예시: 전처리된 데이터 X_reduced, y_labels 사용)
stacking_model.fit(X_train_scaled, y_train)

Exception ignored in: <function Booster.__del__ at 0x0000022297E3DA80>
Traceback (most recent call last):
  File "c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\lightgbm\basic.py", line 3732, in __del__
    _safe_call(_LIB.LGBM_BoosterFree(self._handle))
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: exception: access violation reading 0xFFFFFFFF00000033


,estimators,"[('rf', ...), ('xgb', ...), ...]"
,final_estimator,LogisticRegre...ndom_state=23)
,cv,StratifiedKFo... shuffle=True)
,stack_method,'auto'
,n_jobs,-1
,passthrough,False
,verbose,0
,n_estimators,390
,criterion,'gini'
,max_depth,25
,min_samples_split,7


In [25]:
stack_proba_val = stacking_model.predict_proba(X_val_scaled)[:, 1]

print("\n" + "="*100)
print("=== Stacking Validation 성능 (threshold=0.50) ===")
print_full_metrics("Stacking(LogReg on [RF, XGB, LGB])", y_val, stack_proba_val, threshold=0.5)

stack_best_thr, stack_best_f1, stack_best_rec = find_best_threshold(y_val, stack_proba_val)
print(f"[Stacking] F1 기준 최적 Threshold: {stack_best_thr:.2f}, F1: {stack_best_f1:.4f}, Recall: {stack_best_rec:.4f}")


=== Stacking Validation 성능 (threshold=0.50) ===
===== Stacking(LogReg on [RF, XGB, LGB]) 성능 (Threshold: 0.50) =====
AUC       : 0.8490
정확도    : 0.8187
정밀도    : 0.1424
재현율    : 0.7126
F1-score  : 0.2373

Confusion Matrix:
[[12018  2584]
 [  173   429]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9858    0.8230    0.8971     14602
           1     0.1424    0.7126    0.2373       602

    accuracy                         0.8187     15204
   macro avg     0.5641    0.7678    0.5672     15204
weighted avg     0.9524    0.8187    0.8710     15204


[Stacking] F1 기준 최적 Threshold: 0.79, F1: 0.2950, Recall: 0.4950


In [30]:
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score, confusion_matrix, classification_report

# 1) validation 확률 예측
y_val_proba = stacking_model.predict_proba(X_val_scaled)[:, 1]  # 양성 클래스(1)의 확률

# 2) 기본 threshold = 0.5 로 이진 예측
val_threshold = 0.5
y_val_pred = (y_val_proba >= val_threshold).astype(int)

# 3) 지표 계산
auc_val     = roc_auc_score(y_val, y_val_proba)
f1_val      = f1_score(y_val, y_val_pred)
recall_val  = recall_score(y_val, y_val_pred)
precision_val = precision_score(y_val, y_val_pred, zero_division=0)
cm_val      = confusion_matrix(y_val, y_val_pred)

print(f"\n===== Stacking Validation 성능 (threshold={val_threshold:.2f}) =====")
print(f"Stacking 모델 ROC-AUC   : {auc_val:.4f}")
print(f"Stacking 모델 정밀도    : {precision_val:.4f}")
print(f"Stacking 모델 재현율    : {recall_val:.4f}")
print(f"Stacking 모델 F1-score  : {f1_val:.4f}")
print("\nConfusion Matrix:")
print(cm_val)

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred, digits=4))


===== Stacking Validation 성능 (threshold=0.50) =====
Stacking 모델 ROC-AUC   : 0.8490
Stacking 모델 정밀도    : 0.1424
Stacking 모델 재현율    : 0.7126
Stacking 모델 F1-score  : 0.2373

Confusion Matrix:
[[12018  2584]
 [  173   429]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9858    0.8230    0.8971     14602
           1     0.1424    0.7126    0.2373       602

    accuracy                         0.8187     15204
   macro avg     0.5641    0.7678    0.5672     15204
weighted avg     0.9524    0.8187    0.8710     15204



In [32]:
# F1 기준 최적 threshold 찾기
best_thr, best_f1, best_rec = find_best_threshold(y_val, y_val_proba)
print(f"\n[Stacking] F1 기준 최적 Threshold: {best_thr:.2f}, F1: {best_f1:.4f}, Recall: {best_rec:.4f}")

# 그 threshold로 최종 지표 다시 계산
y_val_pred_best = (y_val_proba >= best_thr).astype(int)

auc_val_best      = roc_auc_score(y_val, y_val_proba)   # AUC는 threshold와 무관
precision_val_best = precision_score(y_val, y_val_pred_best, zero_division=0)
recall_val_best    = recall_score(y_val, y_val_pred_best)
f1_val_best        = f1_score(y_val, y_val_pred_best)
cm_val_best        = confusion_matrix(y_val, y_val_pred_best)

print(f"\n===== Stacking Validation 성능 (best threshold={best_thr:.2f}) =====")
print(f"Stacking 모델 ROC-AUC   : {auc_val_best:.4f}")
print(f"Stacking 모델 정밀도    : {precision_val_best:.4f}")
print(f"Stacking 모델 재현율    : {recall_val_best:.4f}")
print(f"Stacking 모델 F1-score  : {f1_val_best:.4f}")
print("\nConfusion Matrix:")
print(cm_val_best)

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred_best, digits=4))



[Stacking] F1 기준 최적 Threshold: 0.79, F1: 0.2950, Recall: 0.4950

===== Stacking Validation 성능 (best threshold=0.79) =====
Stacking 모델 ROC-AUC   : 0.8490
Stacking 모델 정밀도    : 0.2102
Stacking 모델 재현율    : 0.4950
Stacking 모델 F1-score  : 0.2950

Confusion Matrix:
[[13482  1120]
 [  304   298]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9779    0.9233    0.9498     14602
           1     0.2102    0.4950    0.2950       602

    accuracy                         0.9063     15204
   macro avg     0.5941    0.7092    0.6224     15204
weighted avg     0.9475    0.9063    0.9239     15204



In [26]:
from utils.user_utils import get_model_train_eval, get_model_HO_train_eval

get_model_HO_train_eval(
    model           = rf_clf,
    model_name      = "RF_log1p_HP8",   # 파일 이름에 들어갈 이름
    X_train         = X_train_scaled,
    X_test          = X_val_scaled,
    y_train         = y_train,
    y_test          = y_val,
    hyperopt_params = rf_best_param    # ← HyperOpt best params dict
)

get_model_HO_train_eval(
    model           = xgb_clf,
    model_name      = "XGB_log1p_HP",
    X_train         = X_train_scaled,
    X_test          = X_val_scaled,
    y_train         = y_train,
    y_test          = y_val,
    hyperopt_params = xgb_best_params
)

get_model_HO_train_eval(
    model           = lgbm_clf,
    model_name      = "LGBM_log1p_HP",
    X_train         = X_train_scaled,
    X_test          = X_val_scaled,
    y_train         = y_train,
    y_test          = y_val,
    hyperopt_params = lgbm_best_param
)

✓ 모델 저장 완료: ../models\RF_log1p_HP8.pkl
  파일 크기: 86.13 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8346, 정확도: 0.9597, 정밀도: 0.2963, 재현율: 0.0133, F1: 0.0254
오차행렬:
[[14583    19]
 [  594     8]]
실행 시간: 7.18304181098938
하이퍼파라미터: {'random_state': 23, 'n_estimators': 390, 'max_depth': 25, 'class_weight': {0: 1, 1: 2}, 'min_samples_leaf': 1, 'min_samples_split': 7, 'n_jobs': -1}
✓ 모델 저장 완료: ../models\XGB_log1p_HP.pkl
  파일 크기: 1.08 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8197, 정확도: 0.8979, 정밀도: 0.1890, 재현율: 0.4801, F1: 0.2712
오차행렬:
[[13362  1240]
 [  313   289]]
실행 시간: 1.6525640487670898
하이퍼파라미터: {'random_state': 23, 'n_estimators': 320, 'colsample_bytree': 0.88, 'gamma': 0.058, 'learning_rate': 0.13, 'max_depth': 6, 'scale_pos_weight': 10, 'min_child_weight': 2, 'subsample': 0.85, 'eval_metric': 'auc', 'use_label_encoder': False, 'n_jobs': -1}
✓ 모델 저장 완료: ../models\LGBM_log1p_HP.pkl
  파일 크기: 1.55 MB
folder = c:\big20\git\big20

In [27]:
# ============================================================
# 11. Stacking – RF/XGB/LGB → Logistic Regression
# ============================================================
print("\n" + "="*100)
print("Step11. Stacking (RF + XGB + LGB → Logistic Regression)")
print("="*100)

# Validation 기준 base model 확률값 → 메타 입력
stack_X_val = np.vstack([
    rf_proba_val,
    xgb_proba_val,
    lgb_proba_val
]).T  # (n_val, 3)

meta = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)
meta.fit(stack_X_val, y_val)

coef = meta.coef_[0]
base_models = ["RandomForest", "XGBoost", "LightGBM"]

print("개별 모델 기여도(로지스틱 회귀 계수):")
for name, c in zip(base_models, coef):
    print(f"{name} 기여도(계수): {c:.4f}")

stack_proba_val = meta.predict_proba(stack_X_val)[:, 1]

# (1) 기본 threshold=0.5 성능
print("\n=== Stacking 상세 리포트 (Threshold=0.50) ===")
print_full_metrics("Stacking(LogReg on [RF, XGB, LGB])", y_val, stack_proba_val, threshold=0.5)

# (2) F1 기준 최적 threshold 탐색
stack_best_thr, stack_best_f1, stack_best_rec = find_best_threshold(y_val, stack_proba_val)
stack_best_pred = (stack_proba_val >= stack_best_thr).astype(int)
stack_best_auc  = roc_auc_score(y_val, stack_proba_val)

print("=== Stacking – F1 기준 최적 Threshold 결과 ===")
print(f"Stacking 모델 ROC-AUC: {stack_best_auc:.4f}")
print(f"Stacking 모델 F1-Score: {stack_best_f1:.4f}")
print(f"Stacking 모델 Recall:   {stack_best_rec:.4f}")
print(f"적용 Threshold         : {stack_best_thr:.2f}")

print("\n=== Stacking(최적 Threshold) 상세 리포트 ===")
print_full_metrics(
    f"Stacking(LogReg) [thr={stack_best_thr:.2f}]",
    y_val,
    stack_proba_val,
    threshold=stack_best_thr
)



Step11. Stacking (RF + XGB + LGB → Logistic Regression)
개별 모델 기여도(로지스틱 회귀 계수):
RandomForest 기여도(계수): 4.5220
XGBoost 기여도(계수): -1.5393
LightGBM 기여도(계수): 1.9537

=== Stacking 상세 리포트 (Threshold=0.50) ===
===== Stacking(LogReg on [RF, XGB, LGB]) 성능 (Threshold: 0.50) =====
AUC       : 0.8385
정확도    : 0.8175
정밀도    : 0.1392
재현율    : 0.6960
F1-score  : 0.2319

Confusion Matrix:
[[12010  2592]
 [  183   419]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9850    0.8225    0.8964     14602
           1     0.1392    0.6960    0.2319       602

    accuracy                         0.8175     15204
   macro avg     0.5621    0.7593    0.5642     15204
weighted avg     0.9515    0.8175    0.8701     15204


=== Stacking – F1 기준 최적 Threshold 결과 ===
Stacking 모델 ROC-AUC: 0.8385
Stacking 모델 F1-Score: 0.2748
Stacking 모델 Recall:   0.5100
적용 Threshold         : 0.76

=== Stacking(최적 Threshold) 상세 리포트 ===
===== Stacking(LogReg) [thr=0.76] 성능 (Threshold: 